# 04. Default Model Improvement - Freddie Mac SFLLD

*Planned: extend the baseline default model (`02_default_modeling.ipynb`) with
additional categorical features (property state, occupancy status, loan
purpose) and/or the full (non-sample) dataset to test whether performance
improves beyond the current ROC-AUC ceiling (~0.76-0.77).*

## Summary

This notebook tests three approaches to improving the baseline default model
(`02_default_modeling.ipynb`, ROC-AUC 0.769 random split): adding categorical
features, adding macroeconomic features, and validating with a realistic
time-based (backtest) split. Key findings:

- **Categorical features** (property state, occupancy status, loan purpose,
  one-hot encoded into 60 columns) improved ROC-AUC only marginally, from
  0.7686 to 0.7736.
- **Macroeconomic features** (rate spread, state-level unemployment, HPI
  change — all measured at a fixed 24-month post-origination point) performed
  no better (0.7715), despite literature suggesting these variables are
  meaningful default predictors. This likely reflects a methodological
  limitation: default can occur at any point in a loan's life, so a single
  fixed-point macro snapshot may miss the actual conditions present when a
  given loan's default was triggered — unlike prepayment, where refinancing
  decisions cluster predictably around a known timing window.
- A brief literature review (Campbell & Dietrich 1983; Goodman et al. 2014;
  and others) confirmed that FICO, LTV, and DTI are consistently identified
  as the dominant predictors of mortgage default, with macroeconomic variables
  typically providing only modest incremental value — consistent with what
  this project found empirically.
- **Backtesting** (train on 2018-2020, test on 2021-2022) showed a modest
  performance decline (ROC-AUC 0.7463 vs. 0.7686 under random split),
  reflecting the shift in economic regime between the two periods (default
  rate: 4.28% train vs. 2.41% test). This connects to Sousa et al.'s (2015)
  finding that credit models trained on older data may not fully capture
  regime shifts — though the model still retained meaningful discriminative
  power on genuinely unseen future data.
- **Overall conclusion:** default risk in this dataset is well-explained by
  a small set of core underwriting variables (credit score, interest rate,
  DTI), and neither additional categorical detail nor fixed-point macro
  snapshots meaningfully improve on this. This stands in clear contrast to
  the prepayment model (Project 5), where a single engineered feature
  (rate spread) drove a large improvement — highlighting a genuine difference
  in the underlying drivers of the two outcomes.

## 1. Setup & Imports

In [1]:
import sys

from src.data_loader import clean_orig_data

sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_loader import build_model_dataset, load_orig_data, clean_orig_data, get_default_labels, calculate_state_unemployment, calculate_hpi_change, calculate_rate_spread

## 2. Load 2018-2022 Data (Reusing Existing Pipeline)

Reuse `build_model_dataset()` from the default modeling notebook, which
already loads origination data (including categorical fields) and merges in
the default label. Restricted to 2018-2022 vintages to avoid right-censoring,
consistent with the original default model.

In [2]:
years = range(2018, 2023)

df_list = []
for year in years:
    df_year = build_model_dataset(year)
    df_list.append(df_year)

df_all = pd.concat(df_list, ignore_index=True)

print(df_all.shape)

(250000, 35)


## 3. Inspect Categorical Features Before Encoding

Check the number of unique categories in each candidate categorical feature
to anticipate how many columns one-hot encoding will produce.

In [3]:
categorical_features = ['property_state', 'occupancy_status', 'loan_purpose']

for col in categorical_features:
    print(f"{col}: {df_all[col].nunique()} unique values")
    print(df_all[col].value_counts(dropna=False))
    print()

property_state: 54 unique values
property_state
CA    29125
TX    19749
FL    17310
IL    10504
OH     9316
MI     8972
AZ     8944
GA     8136
NC     7852
PA     7756
NY     7604
CO     7482
WA     7457
VA     7014
NJ     6688
IN     6046
MN     5581
MA     5536
TN     5092
MO     5036
MD     4981
UT     4677
OR     4436
WI     4346
SC     4140
KY     3230
NV     3203
AL     2885
CT     2508
LA     2321
OK     2071
IA     1999
KS     1989
ID     1906
AR     1742
NH     1307
NE     1215
NM     1123
ME     1052
DE      984
MS      935
MT      831
RI      772
WV      625
HI      596
SD      562
DC      514
ND      491
VT      489
AK      418
WY      368
GU       34
PR       31
VI       19
Name: count, dtype: int64

occupancy_status: 3 unique values
occupancy_status
P    224381
I     16628
S      8991
Name: count, dtype: int64

loan_purpose: 3 unique values
loan_purpose
P    126669
N     66558
C     56773
Name: count, dtype: int64



## 4. One-Hot Encode Categorical Features

Convert `property_state`, `occupancy_status`, and `loan_purpose` into one-hot
encoded columns (one binary column per category), which allows these
categorical variables to be used in the model. This has a similar effect to
including fixed effects in a traditional regression — each category gets its
own indicator, letting the model learn category-specific patterns.

In [4]:
categorical_features = ['property_state', 'occupancy_status', 'loan_purpose']

df_encoded = pd.get_dummies(df_all, columns=categorical_features, drop_first = False)

print(df_all.shape)
print(df_encoded.shape)

#Check the new columns that were created
new_columns = [col for col in df_encoded.columns if col not in df_all.columns]
print(f"\n{len(new_columns)} new columns created")
print(new_columns[:10])

(250000, 35)
(250000, 92)

60 new columns created
['property_state_AK', 'property_state_AL', 'property_state_AR', 'property_state_AZ', 'property_state_CA', 'property_state_CO', 'property_state_CT', 'property_state_DC', 'property_state_DE', 'property_state_FL']


## 5. Retrain Model with Expanded Feature Set

Train a Random Forest model using the original 9 numeric features plus the
60 new one-hot encoded categorical columns, and compare performance against
the original baseline (ROC-AUC 0.769 from `02_default_modeling.ipynb`).

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

original_features = [
    'credit_score', 'original_cltv', 'original_dti', 'original_upb',
    'original_ltv', 'original_interest_rate', 'original_loan_term',
    'number_of_borrowers', 'number_of_units'
]

# Combine original numeric features with the new one-hot encoded categorical columns
expanded_features = original_features + list(new_columns)

target = 'default_flag'

df_model_expanded = df_encoded[expanded_features + [target]].dropna()

X_expanded = df_model_expanded[expanded_features]
y_expanded = df_model_expanded[target]

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_expanded, y_expanded, test_size=0.2, random_state=42, stratify=y_expanded
)

rf_model_expanded = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight='balanced', random_state=42
)
rf_model_expanded.fit(X_train_exp, y_train_exp)

rf_pred_proba_expanded = rf_model_expanded.predict_proba(X_test_exp)[:, 1]

print(f"Expanded features - ROC-AUC: {roc_auc_score(y_test_exp, rf_pred_proba_expanded):.4f}")
print(f"Original features (baseline) - ROC-AUC: 0.7686")

Expanded features - ROC-AUC: 0.7736
Original features (baseline) - ROC-AUC: 0.7686


### Marginal Improvement from Categorical Features

Adding 60 one-hot encoded categorical features (property state, occupancy
status, loan purpose) improved ROC-AUC only marginally, from 0.7686 to 0.7736
(+0.005). This is a much smaller gain than the rate_spread feature achieved
for prepayment prediction (+0.067), suggesting two things: (1) these
categorical variables likely carry information that overlaps with existing
numeric features (e.g., state-level risk may already be partially reflected
in credit score or DTI distributions), and (2) unlike prepayment — where
`rate_spread` captured a direct causal driver (refinancing incentive) —
default is likely driven by borrower-specific financial shocks (job loss,
medical expenses, divorce) or macroeconomic conditions (local unemployment,
regional home price declines) that are not observable in static,
origination-time loan characteristics at all. Adding more of the *same kind*
of feature (static, loan-level attributes) appears to have diminishing
returns; a meaningful improvement would likely require fundamentally
different data — e.g., time-varying local economic indicators.

### Literature Context: What Variables Do Industry/Academic Models Typically Use?

A brief literature review confirms that loan-level static features (FICO, LTV,
DTI) — already used in this project — are consistently identified as core
predictors going back to early studies (Campbell and Dietrich, 1983). However,
the literature consistently highlights **macroeconomic variables** —
particularly regional unemployment rate and house price index (HPI) — as an
important complementary category, since they capture borrower financial
shocks and collateral value changes that static loan characteristics cannot.
Notably, one study found unemployment's marginal effect to be modest compared
to established risk factors like leverage and FICO score, suggesting these
macro variables may provide only incremental improvement — consistent with
the diminishing returns observed when adding categorical features in this
project. This suggests HPI (already collected in Project 2 of this portfolio)
is a promising next feature to test.

## 5. Macroeconomic Feature Engineering

Building on literature review findings (Campbell & Dietrich, 1983; Goodman
et al., 2014; and others), extend the feature set beyond static loan
characteristics with three macroeconomic variables measured at a fixed point
after origination (24 months, consistent with the reference point established
in the prepayment analysis):

1. **House Price Index (HPI):** cumulative change in FHFA's national HPI from
   origination to 24 months later, capturing collateral value erosion.
2. **Regional Unemployment Rate:** state-level unemployment rate 24 months
   post-origination (FRED), capturing borrower income shock risk.
3. **Interest Rate Spread:** reusing the `rate_spread` feature from the
   prepayment model, since rising rates can also signal a broader tightening
   economic environment correlated with default risk.

This tests whether macroeconomic conditions add meaningful predictive power
beyond loan-level characteristics, as suggested by prior research — while
also acknowledging that some studies find such effects to be modest relative
to core underwriting variables like FICO and LTV.

## 6. Load State-Level Unemployment Data (FRED API)

Retrieve monthly state-level unemployment rates for all 50 states + DC via
the FRED API, using `fredapi` and a `.env`-stored API key (excluded from git).
This provides time-varying, geography-specific economic shock information
that static state dummy variables cannot capture.

## Environment Variables

This project uses the FRED API to retrieve macroeconomic data. To reproduce:
1. Obtain a free API key from https://fred.stlouisfed.org/docs/api/api_key.html
2. Create a `.env` file in the project root with: `FRED_API_KEY=your_key_here`
3. This file is excluded from git via `.gitignore` and should never be committed.

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

fred_api_key = os.getenv('FRED_API_KEY')

from fredapi import Fred
fred = Fred(api_key=fred_api_key)

# US state abbreviations (matches property_state values in our data)
state_codes = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'DC', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA',
    'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY',
    'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX',
    'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'
]

unemployment_list = []

for state in state_codes:
    series_id = f'{state}UR'
    try:
        series = fred.get_series(series_id)
        df_state = series.reset_index()
        df_state.columns = ['date', 'unemployment_rate']
        df_state['state'] = state
        unemployment_list.append(df_state)
    except Exception as e:
        print(f"Failed for {state}: {e}")

unemployment_all = pd.concat(unemployment_list, ignore_index=True)
unemployment_all.to_csv('../data/raw/fred/state_unemployment.csv', index=False)

print(unemployment_all.shape)
print(unemployment_all['state'].nunique())

(30855, 3)
51


## 7. Calculate State Unemployment at 24 Months Post-Origination

For each loan, look up the unemployment rate in its property state, 24 months
after origination (consistent with the reference point used for rate spread),
following the same logic as `calculate_rate_spread()`.

In [7]:
unemployment_all = pd.read_csv('../data/raw/fred/state_unemployment.csv')
unemployment_all['date'] = pd.to_datetime(unemployment_all['date'])
unemployment_all['year_month'] = unemployment_all['date'].dt.strftime('%Y%m').astype(int)

# Test on 2018 data
df_orig_2018 = load_orig_data(2018)
df_orig_2018 = clean_orig_data(df_orig_2018)

df_with_unemployment_2018 = calculate_state_unemployment(df_orig_2018, unemployment_all)

print(df_with_unemployment_2018[['loan_sequence_number', 'property_state', 'first_payment_date', 'reference_year_month', 'unemployment_rate']].head())
print(df_with_unemployment_2018['unemployment_rate'].isna().sum())

  loan_sequence_number property_state  first_payment_date  \
0         F18Q10000028             KY              201803   
1         F18Q10000052             MI              201803   
2         F18Q10000084             IA              201803   
3         F18Q10000117             IA              201803   
4         F18Q10000140             IA              201803   

   reference_year_month  unemployment_rate  
0                202003                4.1  
1                202003                3.7  
2                202003                2.7  
3                202003                2.7  
4                202003                2.7  
20


## 8. Combine All Macroeconomic Features Across Vintage Years

Calculate rate spread, state unemployment, and national HPI change (all at
24 months post-origination) for all 2018-2022 vintage loans, then merge with
the default label to build the final expanded modeling dataset.

In [8]:
fhfa = pd.read_excel(
    '../data/raw/fhfa/hpi_at_national.xlsx',
    skiprows=5,
    names=['year', 'annual_chg_pct', 'hpi', 'hpi_1990_base', 'hpi_2000_base']
)

market_rate = pd.read_csv('../data/raw/fred/MORTGAGE30US.csv')

market_rate = market_rate.rename(columns={
    'observation_date': 'rate_date',
    'MORTGAGE30US': 'market_rate'
})
market_rate['rate_date'] = pd.to_datetime(market_rate['rate_date'])
market_rate['year_month'] = market_rate['rate_date'].dt.strftime('%Y%m').astype(int)

In [9]:
years = range(2018, 2023)

df_macro_list = []
for year in years:
    df_orig = load_orig_data(year)
    df_orig = clean_orig_data(df_orig)
    df_orig['vintage_year'] = year

    df_orig = calculate_rate_spread(df_orig, market_rate)
    df_orig = calculate_state_unemployment(df_orig, unemployment_all)
    df_orig = calculate_hpi_change(df_orig, fhfa)

    df_macro_list.append(df_orig)

df_all_macro = pd.concat(df_macro_list, ignore_index=True)

# Merge in default label
default_labels_all = []
for year in years:
    default_labels_all.append(get_default_labels(year))
default_labels_all_df = pd.concat(default_labels_all, ignore_index=True)

df_all_macro = df_all_macro.merge(default_labels_all_df, on='loan_sequence_number', how='left')

print(df_all_macro.shape)
print(df_all_macro[['rate_spread', 'unemployment_rate', 'hpi_change_pct']].isna().sum())

(250000, 49)
rate_spread           0
unemployment_rate    85
hpi_change_pct        4
dtype: int64


### Macro Feature Integration Summary

Successfully merged rate spread, state unemployment, and HPI change into the
2018-2022 dataset (250,000 loans, 49 columns). Missing values are minimal:
`unemployment_rate` has 85 missing (likely U.S. territories like Guam, Puerto
Rico, and the Virgin Islands, which lack FRED state unemployment series), and
`hpi_change_pct` has 4 missing (edge cases near the data range boundary).
These will be dropped during modeling with negligible impact on sample size.

## 9. Merge Default Label and Retrain Model with Macro Features

Merge the default label, then train a Random Forest model using the original
numeric features plus the three new macroeconomic features (rate spread,
state unemployment, HPI change), and compare against both the original
baseline (ROC-AUC 0.769) and the categorical-features-only model (ROC-AUC
0.774).

In [13]:
default_labels_all = []
for year in years:
    default_labels_all.append(get_default_labels(year))
default_labels_all_df = pd.concat(default_labels_all, ignore_index=True)

if 'default_flag' in df_all_macro.columns:
    df_all_macro = df_all_macro.drop(columns=['default_flag'])

df_all_macro = df_all_macro.merge(default_labels_all_df, on='loan_sequence_number', how='left')

macro_features = original_features + ['rate_spread', 'unemployment_rate', 'hpi_change_pct']

df_model_macro = df_all_macro[macro_features + [target]].dropna()

X_macro = df_model_macro[macro_features]
y_macro = df_model_macro[target]

X_train_macro, X_test_macro, y_train_macro, y_test_macro = train_test_split(
    X_macro, y_macro, test_size=0.2, random_state=42, stratify=y_macro
)

rf_model_macro = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight='balanced', random_state=42
)
rf_model_macro.fit(X_train_macro, y_train_macro)

rf_pred_proba_macro = rf_model_macro.predict_proba(X_test_macro)[:, 1]

print(f"Macro features - ROC-AUC: {roc_auc_score(y_test_macro, rf_pred_proba_macro):.4f}")
print(f"Categorical features (previous) - ROC-AUC: 0.7736")
print(f"Original baseline - ROC-AUC: 0.7686")

Macro features - ROC-AUC: 0.7715
Categorical features (previous) - ROC-AUC: 0.7736
Original baseline - ROC-AUC: 0.7686


### Macro Features Show No Improvement Over Categorical Features

Adding macroeconomic features (rate spread, state unemployment, HPI change —
all measured at a fixed 24-month post-origination point) resulted in ROC-AUC
0.7715, essentially unchanged from the categorical-features model (0.7736)
and only marginally above the original baseline (0.7686).

This is consistent with prior literature noting that macroeconomic variables'
marginal contribution to default prediction is often modest relative to
established underwriting variables like FICO and leverage. It may also reflect
a methodological limitation specific to this approach: unlike prepayment
(where refinancing decisions cluster predictably around a known timing
window), default can occur at any point in a loan's life. Measuring macro
conditions at a single fixed point (24 months) may miss the actual economic
conditions present at the time a given loan's default was triggered — a loan
that defaulted at month 8 due to a local unemployment spike would not be well
represented by its 24-month unemployment reading.

A more rigorous approach would require tracking macro conditions as a
time-varying input throughout each loan's life (e.g., unemployment rate in
the month immediately preceding delinquency), which would require restructuring
this as a time-varying/survival framework rather than the current fixed-label,
fixed-snapshot approach — a natural extension for future work, but beyond the
current project's binary-label modeling structure.

## 10. Backtest: Time-Based Train/Test Split

Instead of a random 80/20 split, train on 2018-2020 vintages and test on
2021-2022 vintages — loans the model has never seen from a future time period
it wasn't trained on. This more closely simulates real-world deployment, where
a model must generalize to loans originated after it was built. Uses the
original 9 numeric baseline features to isolate the effect of the validation
method itself, separate from the feature-engineering experiments above.

In [14]:
# Time-based split: train on 2018-2020, test on 2021-2022
df_baseline_time = df_all_macro[original_features + [target, 'vintage_year']].dropna()

train_mask = df_baseline_time['vintage_year'] <= 2020
test_mask = df_baseline_time['vintage_year'] > 2020

X_train_time = df_baseline_time.loc[train_mask, original_features]
y_train_time = df_baseline_time.loc[train_mask, target]
X_test_time = df_baseline_time.loc[test_mask, original_features]
y_test_time = df_baseline_time.loc[test_mask, target]

print(f"Train set: {X_train_time.shape[0]} loans (2018-2020)")
print(f"Test set: {X_test_time.shape[0]} loans (2021-2022)")
print(f"Train default rate: {y_train_time.mean():.2%}")
print(f"Test default rate: {y_test_time.mean():.2%}")

rf_model_time = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight='balanced', random_state=42
)
rf_model_time.fit(X_train_time, y_train_time)

rf_pred_proba_time = rf_model_time.predict_proba(X_test_time)[:, 1]

print(f"\nTime-based split - ROC-AUC: {roc_auc_score(y_test_time, rf_pred_proba_time):.4f}")
print(f"Random split (original baseline) - ROC-AUC: 0.7686")

Train set: 149413 loans (2018-2020)
Test set: 99979 loans (2021-2022)
Train default rate: 4.28%
Test default rate: 2.41%

Time-based split - ROC-AUC: 0.7463
Random split (original baseline) - ROC-AUC: 0.7686


### Backtest Results: Modest Performance Decline, As Expected

Time-based validation (train on 2018-2020, test on 2021-2022) yields ROC-AUC
0.7463, a modest decline from the random-split baseline (0.7686). This decline
is expected and reflects a more realistic evaluation: the random split allowed
the model to implicitly learn from patterns present in both the 2021-2022
period (via other loans from the same vintages in the training set) —
information a model would not have in real deployment.

The decline is also consistent with the substantial shift in default rate
between the train period (4.28%) and test period (2.41%), reflecting the
same regime change (COVID-era underwriting/rate conditions) discussed
throughout this project. This connects directly to the Sousa et al. (2015)
STM/LTM findings referenced earlier: models trained on older data may not
fully capture shifts in economic regime, and this